In [1]:
!pip install -q langchain langchain-core langchain-community
!pip install -q langchain-groq langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 137.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 195.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 96.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 79.5 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LangSmith")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume Screening System"

In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [6]:
job_description = """
Hiring for Data Scientist role.

Required Skills:
Python
Machine Learning
SQL
Pandas
NumPy
Deep Learning
Data Visualization

Preferred Tools:
TensorFlow
PyTorch
Scikit-learn
Power BI

Experience:
2+ years experience in data analysis or ML projects.
"""

In [7]:
strong_resume = """
John Doe
3 years experience as Data Scientist.
Skills: Python, SQL, Machine Learning, Deep Learning, Pandas, NumPy, Scikit-learn, TensorFlow, Power BI
Built NLP and prediction systems.
"""

average_resume = """
Sarah Khan
1 year internship in analytics.
Skills: Python, SQL, Pandas, Excel, Data Visualization
Worked on dashboards and reports.
"""

weak_resume = """
Alex Roy
Fresher candidate.
Skills: MS Word, Communication, HTML, CSS
No relevant ML experience.
"""

In [8]:
extract_prompt = PromptTemplate.from_template("""
You are an AI recruiter.

Extract from the resume:

1. Skills
2. Experience
3. Tools

Rules:
- Only use information present in resume
- Do not assume anything
- Keep output clean

Resume:
{resume}
""")

In [9]:
score_prompt = PromptTemplate.from_template("""
You are an AI Resume Evaluator.

Compare the candidate resume data with the job description.

Job Description:
{jd}

Candidate Details:
{candidate}

Tasks:
1. Give Fit Score from 0 to 100
2. Explain why this score was assigned
3. Mention missing skills
4. Final recommendation: Strong / Average / Weak

Return clean formatted output.
""")

In [10]:
extract_chain = extract_prompt | llm | StrOutputParser()
score_chain = score_prompt | llm | StrOutputParser()

In [11]:
def screen_resume(resume_text, name):
    print("="*60)
    print("Candidate:", name)

    extracted = extract_chain.invoke({"resume": resume_text})

    result = score_chain.invoke({
        "jd": job_description,
        "candidate": extracted
    })

    print(result)
    print("="*60)

In [12]:
screen_resume(strong_resume, "Strong Candidate")

Candidate: Strong Candidate
**Resume Evaluation Report**

**Candidate Name:** (Not provided)

**Job Description:** Data Scientist
**Hiring Requirements:**

* Required Skills:
	+ Python
	+ Machine Learning
	+ SQL
	+ Pandas
	+ NumPy
	+ Deep Learning
	+ Data Visualization
* Preferred Tools:
	+ TensorFlow
	+ PyTorch
	+ Scikit-learn
	+ Power BI
* Experience:
	+ 2+ years experience in data analysis or ML projects

**Candidate Details:**

* **Skills:**
	+ Python
	+ SQL
	+ Machine Learning
	+ Deep Learning
	+ Pandas
	+ NumPy
	+ Scikit-learn
	+ TensorFlow
	+ Power BI
* **Experience:**
	+ 3 years experience as Data Scientist
* **Tools/Built Systems:**
	+ NLP systems
	+ Prediction systems

**Evaluation Results:**

* **Fit Score:** 92/100
* **Explanation:** The candidate's resume matches the job description closely, with all required skills mentioned and some preferred tools listed. However, there is a minor mismatch in the experience requirement, where the candidate has 3 years of experience, but

In [13]:
screen_resume(average_resume, "Average Candidate")

Candidate: Average Candidate
**Resume Evaluation Report**

**Fit Score:** 44

**Explanation:** The candidate's resume shows some alignment with the job description, but there are significant gaps in required skills and experience.

**Missing Skills:**

1. Machine Learning
2. NumPy
3. Deep Learning
4. TensorFlow
5. PyTorch
6. Scikit-learn
7. Power BI
8. Experience with 2+ years in data analysis or ML projects (candidate has only 1 year internship)

**Additional Observations:**

1. The candidate has listed Excel, which is not mentioned in the job description. While Excel is a useful skill, it is not a priority for this Data Scientist role.
2. The candidate has used Dashboards and Reports in their previous experience, which is somewhat related to Data Visualization, but it is not a specific tool required by the job description.

**Final Recommendation:** Average

The candidate's resume shows some potential, but they are far from meeting the required skills and experience for the Data Scie

In [14]:
screen_resume(weak_resume, "Weak Candidate")

Candidate: Weak Candidate
**Resume Evaluation Report**

**Job Title:** Data Scientist
**Candidate Details:**

**Skills:**
- MS Word
- Communication
- HTML
- CSS

**Experience:**
- No relevant experience mentioned.

**Tools:**
- No specific tools mentioned.

**Evaluation:**

**Fit Score:** 16
This score is assigned due to the significant mismatch between the candidate's skills and experience and the job requirements.

**Reasoning:**
- The candidate lacks essential technical skills required for the Data Scientist role, such as Python, Machine Learning, SQL, Pandas, NumPy, and Deep Learning.
- The candidate's skills are more aligned with non-technical areas, including MS Word and Communication.
- There is no relevant experience mentioned in the candidate's resume.

**Missing Skills:**
- Python
- Machine Learning
- SQL
- Pandas
- NumPy
- Deep Learning
- Data Visualization
- TensorFlow
- PyTorch
- Scikit-learn
- Power BI

**Final Recommendation:** Weak
The candidate's resume does not demons